# 04 — Ghorbani Dataset Analysis

Comprehensive analysis of the Ghorbani Grip Force Prediction dataset from:

> Ghorbani, A., Yousefi-Koma, A., & Vedadi, A. (2023).
> "Estimation and Early Prediction of Grip Force Based on sEMG Signals and Deep Recurrent Neural Networks."
> Journal of the Brazilian Society of Mechanical Sciences and Engineering.
> https://arxiv.org/abs/2302.09555

## Dataset Overview
- **Subjects**: 10 healthy adults (9 male, 1 female, avg age 23.8)
- **EMG Device**: Myo armband (8 channels @ 200 Hz)
- **Force Sensor**: ATI Mini45 F/T sensor (Z-axis grip force)
- **Task**: Precision-type pinch grip
- **Trials**: 3 recordings per subject

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_GHORBANI_DIR, GHORBANI_FS, GHORBANI_CHANNEL_NAMES,
    GHORBANI_SUBJECTS, GHORBANI_EMG_CHANNELS,
    BANDPASS_LOW, BANDPASS_HIGH, BANDPASS_ORDER,
    ENVELOPE_WINDOW_MS, ENVELOPE_HOP_MS,
    MRMR_N_SELECT,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

## 1. Load Ghorbani Dataset — All Subjects

In [ ]:
from src.data.load_ghorbani import load_all_ghorbani, validate_ghorbani_data, print_ghorbani_summary

# Load all subjects
subject_data = load_all_ghorbani(verbose=True)

# Show dataset summary
print_ghorbani_summary(subject_data)

In [ ]:
# Load first subject for detailed exploration
subj = subject_data[0]
emg = subj['emg']
force = subj['force']

print(f"Subject ID: {subj['subject_id']}")
print(f"EMG shape: {emg.shape}  (samples x channels)")
print(f"Force shape: {force.shape}")
print(f"Duration: {len(emg) / GHORBANI_FS:.1f} seconds")
print(f"Sampling rate: {GHORBANI_FS} Hz")

## 2. Raw EMG Signal Visualization

In [ ]:
# Plot first 5 seconds of all 8 Myo channels
t_show = 5  # seconds
n_show = int(t_show * GHORBANI_FS)
t = np.arange(n_show) / GHORBANI_FS

fig, axes = plt.subplots(4, 2, figsize=(16, 12), sharex=True)
for i, ax in enumerate(axes.flat):
    ax.plot(t, emg[:n_show, i], linewidth=0.4, color='steelblue')
    ax.set_ylabel(GHORBANI_CHANNEL_NAMES[i], fontsize=8)
    ax.grid(alpha=0.3)
    
axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')
fig.suptitle('Myo Armband EMG — Subject 1 (first 5 seconds)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Force Signal Analysis

In [ ]:
# Plot total grip force
t_all = np.arange(len(force)) / GHORBANI_FS

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_all, force, linewidth=0.6, color='darkorange')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Grip Force (N)')
ax.set_title('ATI Mini45 Z-axis Grip Force — Subject 1 (full recording)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Force range: [{force.min():.2f}, {force.max():.2f}] N")
print(f"Force mean: {force.mean():.2f} N")
print(f"Force std: {force.std():.2f} N")

In [ ]:
# Force distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# All subjects force distribution
all_forces = np.concatenate([s['force'] for s in subject_data])
axes[0].hist(all_forces, bins=100, color='darkorange', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Grip Force (N)')
axes[0].set_ylabel('Count')
axes[0].set_title('Force Distribution (All Subjects)')

# Per-subject force boxplot
force_per_subj = [s['force'] for s in subject_data]
bp = axes[1].boxplot(force_per_subj, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
axes[1].set_xlabel('Subject ID')
axes[1].set_ylabel('Grip Force (N)')
axes[1].set_title('Force Range per Subject')
axes[1].set_xticklabels([str(s['subject_id']) for s in subject_data])

plt.tight_layout()
plt.show()

## 4. EMG + Force Temporal Alignment

In [ ]:
# Plot EMG envelope and force together
from src.data.preprocess import compute_rms_envelope

# Compute RMS envelope
envelope = compute_rms_envelope(emg, ENVELOPE_WINDOW_MS, ENVELOPE_HOP_MS, GHORBANI_FS)

# Downsample force to match envelope
win_samp = int(ENVELOPE_WINDOW_MS * GHORBANI_FS / 1000)
hop_samp = int(ENVELOPE_HOP_MS * GHORBANI_FS / 1000)
n_windows = len(envelope)
force_env = np.zeros(n_windows)
for i in range(n_windows):
    start = i * hop_samp
    end = min(start + win_samp, len(force))
    force_env[i] = np.mean(force[start:end])

t_env = np.arange(len(envelope)) * ENVELOPE_HOP_MS / 1000

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

# EMG Channel 0 envelope
axes[0].plot(t_env, envelope[:, 0], color='steelblue', linewidth=0.8)
axes[0].set_ylabel(GHORBANI_CHANNEL_NAMES[0])
axes[0].set_title('EMG RMS Envelope')

# EMG Channel 4 envelope (extensor region)
axes[1].plot(t_env, envelope[:, 4], color='seagreen', linewidth=0.8)
axes[1].set_ylabel(GHORBANI_CHANNEL_NAMES[4])

# Force envelope
axes[2].plot(t_env, force_env, color='darkorange', linewidth=0.8)
axes[2].set_ylabel('Grip Force (N)')
axes[2].set_xlabel('Time (s)')
axes[2].set_title('Target Force')

plt.tight_layout()
plt.show()

print(f"Envelope shape: {envelope.shape}")
print(f"Effective envelope rate: {1000 / ENVELOPE_HOP_MS:.1f} Hz")

## 5. Channel Correlation Analysis

In [ ]:
# Compute correlation matrix between all EMG channels and force
# Using envelope data

# Stack all subjects' envelopes
all_envelopes = []
all_forces_env = []

for subj in subject_data:
    env = compute_rms_envelope(subj['emg'], ENVELOPE_WINDOW_MS, ENVELOPE_HOP_MS, GHORBANI_FS)
    n_win = len(env)
    f_env = np.zeros(n_win)
    for i in range(n_win):
        start = i * hop_samp
        end = min(start + win_samp, len(subj['force']))
        f_env[i] = np.mean(subj['force'][start:end])
    
    all_envelopes.append(env)
    all_forces_env.append(f_env)

X_all = np.concatenate(all_envelopes, axis=0)
y_all = np.concatenate(all_forces_env)

# Create combined matrix for correlation
combined = np.column_stack([X_all, y_all])
labels = GHORBANI_CHANNEL_NAMES + ['Force']

corr_matrix = np.corrcoef(combined.T)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
ax.set_title('Channel-Force Correlation Matrix')
plt.colorbar(im, ax=ax, label='Pearson r')

# Add correlation values as text
for i in range(len(labels)):
    for j in range(len(labels)):
        text = ax.text(j, i, f'{corr_matrix[i, j]:.2f}',
                       ha='center', va='center', fontsize=8,
                       color='white' if abs(corr_matrix[i, j]) > 0.5 else 'black')

plt.tight_layout()
plt.show()

In [ ]:
# Channel-Force correlation ranking
force_corr = corr_matrix[-1, :-1]  # Correlation with force (excluding force itself)

# Sort by absolute correlation
sorted_idx = np.argsort(np.abs(force_corr))[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if c > 0 else 'coral' for c in force_corr[sorted_idx]]
bars = ax.barh([GHORBANI_CHANNEL_NAMES[i] for i in sorted_idx], 
               force_corr[sorted_idx], color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Correlation with Grip Force')
ax.set_title('EMG Channel Relevance to Force (Pearson Correlation)')
ax.set_xlim(-1, 1)

# Add value labels
for bar, val in zip(bars, force_corr[sorted_idx]):
    ax.text(val + 0.02 if val > 0 else val - 0.08, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nChannel ranking by absolute correlation with force:")
for rank, idx in enumerate(sorted_idx, 1):
    print(f"  {rank}. {GHORBANI_CHANNEL_NAMES[idx]}: r = {force_corr[idx]:.4f}")

## 6. MRMR Channel Selection

In [ ]:
from src.features.mrmr import run_mrmr_analysis

# Run MRMR to select best 2 channels
print(f"Running MRMR to select {MRMR_N_SELECT} channels from {GHORBANI_EMG_CHANNELS}...")

mrmr_result = run_mrmr_analysis(
    subject_data, 
    channel_names=GHORBANI_CHANNEL_NAMES,
    n_select=MRMR_N_SELECT,
    train_ratio=0.8  # Only use 80% for channel selection to prevent leakage
)

print(f"\nSelected channels: {mrmr_result['selected_names']}")
print(f"Selected indices: {mrmr_result['selected_indices']}")

In [ ]:
# Visualize MRMR results
relevance = mrmr_result['relevance']
mrmr_scores = mrmr_result['all_channel_mrmr_scores']
redundancy = mrmr_result['redundancy_matrix']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Relevance (MI with force)
sorted_rel_idx = np.argsort(relevance)[::-1]
colors1 = ['forestgreen' if i in mrmr_result['selected_indices'] else 'steelblue' 
           for i in sorted_rel_idx]
axes[0].barh([GHORBANI_CHANNEL_NAMES[i] for i in sorted_rel_idx],
             relevance[sorted_rel_idx], color=colors1, edgecolor='black')
axes[0].set_xlabel('Mutual Information with Force')
axes[0].set_title('Channel Relevance (MI)')

# 2. MRMR scores
sorted_mrmr_idx = np.argsort(mrmr_scores)[::-1]
colors2 = ['forestgreen' if i in mrmr_result['selected_indices'] else 'steelblue' 
           for i in sorted_mrmr_idx]
axes[1].barh([GHORBANI_CHANNEL_NAMES[i] for i in sorted_mrmr_idx],
             mrmr_scores[sorted_mrmr_idx], color=colors2, edgecolor='black')
axes[1].set_xlabel('MRMR Score')
axes[1].set_title('Final MRMR Ranking')

# 3. Redundancy heatmap
im = axes[2].imshow(redundancy, cmap='YlOrRd')
axes[2].set_xticks(range(GHORBANI_EMG_CHANNELS))
axes[2].set_yticks(range(GHORBANI_EMG_CHANNELS))
axes[2].set_xticklabels(GHORBANI_CHANNEL_NAMES, rotation=45, ha='right', fontsize=8)
axes[2].set_yticklabels(GHORBANI_CHANNEL_NAMES, fontsize=8)
axes[2].set_title('Channel Redundancy (Pairwise MI)')
plt.colorbar(im, ax=axes[2], label='MI')

# Highlight selected channels
for idx in mrmr_result['selected_indices']:
    axes[2].axhline(idx - 0.5, color='lime', linewidth=2)
    axes[2].axhline(idx + 0.5, color='lime', linewidth=2)
    axes[2].axvline(idx - 0.5, color='lime', linewidth=2)
    axes[2].axvline(idx + 0.5, color='lime', linewidth=2)

plt.tight_layout()
plt.show()

print("\nMRMR Selection Steps:")
for step in mrmr_result['mrmr_scores']:
    print(f"  Step {step['step']}: Selected {step['selected_name']} "
          f"(MRMR score: {step['mrmr_score']:.4f}, Relevance: {step['relevance']:.4f})")

## 7. Cross-Subject Variability Analysis

In [ ]:
# Analyze EMG amplitude distribution across subjects
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ch_idx, ax in enumerate(axes.flat):
    channel_data = []
    for subj in subject_data:
        # Compute RMS for this channel
        rms_val = np.sqrt(np.mean(subj['emg'][:, ch_idx]**2))
        channel_data.append(rms_val)
    
    ax.bar(range(1, len(subject_data)+1), channel_data, color='steelblue', edgecolor='black')
    ax.set_xlabel('Subject')
    ax.set_ylabel('RMS Amplitude')
    ax.set_title(GHORBANI_CHANNEL_NAMES[ch_idx])
    ax.set_xticks(range(1, len(subject_data)+1))

plt.suptitle('Cross-Subject EMG Amplitude Variability', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Per-subject recording statistics
stats = []
for subj in subject_data:
    stats.append({
        'Subject': subj['subject_id'],
        'Samples': len(subj['force']),
        'Duration (s)': len(subj['force']) / GHORBANI_FS,
        'Force Mean': subj['force'].mean(),
        'Force Std': subj['force'].std(),
        'Force Max': subj['force'].max(),
        'EMG RMS Mean': np.sqrt(np.mean(subj['emg']**2)),
        'Trials': subj.get('n_trials', 3),
    })

df_stats = pd.DataFrame(stats)
df_stats.style.format({
    'Duration (s)': '{:.1f}',
    'Force Mean': '{:.2f}',
    'Force Std': '{:.2f}',
    'Force Max': '{:.2f}',
    'EMG RMS Mean': '{:.4f}',
})

## 8. Comparison with Paper Results

The original paper by Ghorbani et al. (2023) reported:
- GRU: R² = 0.994
- LSTM: R² = 0.992
- MLP: R² = 0.929

They used all 8 EMG channels. Our approach:
- Uses MRMR to select only 2 channels
- Maintains temporal block splitting to prevent data leakage
- Aims to achieve comparable performance with reduced channels

In [ ]:
# Summary
print("=" * 60)
print("GHORBANI DATASET ANALYSIS SUMMARY")
print("=" * 60)
print(f"Total subjects: {len(subject_data)}")
print(f"EMG channels: {GHORBANI_EMG_CHANNELS} (Myo armband)")
print(f"Sampling rate: {GHORBANI_FS} Hz")
print(f"Total samples: {sum(len(s['force']) for s in subject_data):,}")
print(f"\nSelected MRMR channels ({MRMR_N_SELECT}):")
for i, (idx, name) in enumerate(zip(mrmr_result['selected_indices'], mrmr_result['selected_names']), 1):
    print(f"  {i}. {name} (index {idx}, relevance: {relevance[idx]:.4f})")
print(f"\nReady for GRU training with 2-channel input.")
print("=" * 60)